# 第二部分：巴西电商用户行为分析 - 特征工程
## 2.1加载清洗后的数据
## 2.2表连接
#### 2.2.1构建客户——订单表
用于分析客户下单频次、订单状态分布、下单时间趋势等
#### 2.2.2构建订单明细表
用于分析商品销售细节、支付方式偏好、评分与商品关系等
#### 2.2.3构建商品-商家表
用于分析品类销售排行、商家配送效率、商品描述与销量关系等
### 2.2.4保存三个表 用于后续分析
## 2.3构建用户维度特征
#### 2.3.1每个用户的订单数和复购情况
#### 2.3.2每个用户的总消费金额和平均客单价
#### 2.3.3每个用户的首次/最后一次购买时间
#### 2.3.4合并且保存用户维度特征表
用于后续分析
## 2.4构建订单维度特征
#### 2.4.1每个订单的总金额
#### 2.4.2每个订单的商品件数
#### 2.4.3每个订单的配送时长
#### 2.4.4每个订单的延迟情况
#### 2.4.5合并且保存订单维度特征表
用于后续分析
## 2.5构建产品维度特征
#### 2.5.1每个类别产品的销售额（不含运费）和销量
销售额（不含运费）反映产品本身的市场价值，用于判断哪些品类最赚钱、销售贡献最大
#### 2.5.2每个类别产品的平均评分
#### 2.5.3每个产品类别中销售额前10的城市
#### 2.5.4合并且保存产品维度特征表
用于后续分析
## 2.6特征工程总结

#####

## 2.特征工程

In [2]:
import pandas as pd
import numpy as np

#####

## 2.1 加载清洗后的数据

In [323]:
customers = pd.read_csv('brazil_ecommerce_project/olist_customers_dataset.csv')  #客户数据

In [324]:
geolocation = pd.read_csv('brazil_ecommerce_project/olist_geolocation_dataset.csv') #地理位置数据

In [325]:
order_items = pd.read_csv('brazil_ecommerce_project/olist_order_items_dataset.csv'
                          ,parse_dates=['shipping_limit_date']) #订单项目数据

In [326]:
order_payments = pd.read_csv('brazil_ecommerce_project/olist_order_payments_dataset.csv') #订单付款数据

In [327]:
order_reviews = pd.read_csv('brazil_ecommerce_project/olist_order_reviews_dataset.csv'
                            ,parse_dates=['review_creation_date','review_answer_timestamp']) 
#订单评分数据

In [328]:
orders = pd.read_csv('brazil_ecommerce_project/cleaned_data/cleaned_orders.csv'
                    ,parse_dates=['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']) #订单数据

In [329]:
products = pd.read_csv('brazil_ecommerce_project/cleaned_data/cleaned_products.csv') #产品数据

In [330]:
sellers = pd.read_csv('brazil_ecommerce_project/olist_sellers_dataset.csv')  #卖家数据

In [331]:
product_category= pd.read_csv('brazil_ecommerce_project/product_category_name_translation.csv') 
#商品类别名称翻译映射表

#####

## 2.2表连接

### 2.2.1构建客户—订单表
用于分析客户下单频次、订单状态分布、下单时间趋势等

In [332]:
# 直接合并 customers 和 orders（每个客户可能有多个订单）
customer_order = customers.merge(orders, on='customer_id', how='left')

### 2.2.2构建订单明细表
用于分析商品销售细节、支付方式偏好、评分与商品关系等

In [333]:
# 1.先合并 orders 和 order_items（1个订单对应多个商品）
order_merge  = orders.merge(order_items, on='order_id', how='left')

In [334]:
# 2.连接订单支付表（1个订单对应多个支付记录）
order_merge  = order_merge.merge(order_payments, on='order_id', how='left')

In [335]:
# 3.连接订单评分表（1个订单对应1条/多条评价）
order_merge = order_merge.merge(order_reviews, on='order_id', how='left')

### 2.2.3构建商品-商家表
用于分析品类销售排行、商家配送效率、商品描述与销量关系等

In [336]:
# 1.先合并order_items和products（多个订单商品对应一个产品）
product_seller_merge = order_items.merge(products, on='product_id', how='left')

In [337]:
# 2.连接sellers表(多个订单商品对应一个商家）
product_seller_merge = product_seller_merge.merge(sellers, on='seller_id', how='left')

In [338]:
# 3.连接product_category表（多个订单商品对应一个产品类别）
product_seller_merge = product_seller_merge.merge(product_category, 
                                                  on='product_category_name', how='left')

#####

### 2.2.4保存三个表 用于后续分析

In [339]:
customer_order.to_csv('brazil_ecommerce_project/merges/customer_order_merge.csv',index=False)
order_merge.to_csv('brazil_ecommerce_project/merges/order_merge.csv',index=False)
product_seller_merge.to_csv('brazil_ecommerce_project/merges/product_seller_merge.csv',index=False)

#####

## 2.3构建用户维度特征

### 2.3.1每个用户的订单数和复购情况

In [340]:
#每个用户的订单数
cust_count = customer_order.groupby('customer_id')['order_id'].nunique().reset_index()
#用布尔特征标记复购的情况
cust_count['is_repeat'] = cust_count['order_id'] > 1
cust_count.head()

,customer_id,order_id,is_repeat
0,00012a2ce6f8dcda20d059ce98491703,1,False
1,000161a058600d5901f007fab4c27140,1,False
2,0001fd6190edaaf884bcaf3d49edf079,1,False
3,0002414f95344307404f0ace7a26f1d5,1,False
4,000379cdec625522490c315e70c7a9fb,1,False


In [341]:
#索引顾客复购的订单
cust_count[cust_count['is_repeat'] == True]

,customer_id,order_id,is_repeat


#### 结论：客户只下过一单，没有复购情况

### 2.3.2每个用户的总消费金额和平均客单价

In [342]:
# 1. 每个用户的总消费金额：
# 用 nunique()统计用户的订单数，因为订单明细表中一个订单会对应多行商品记录,nunique() 能去重得到用户真实的下单次数；
# 如果用 count()，会把商品行数当成订单数，导致客单价被严重低估，不符合业务逻辑。
total_spent_count = order_merge.groupby('customer_id').agg(total_spent=('payment_value', 'sum'),
                                       order_count=('order_id', 'nunique')).reset_index()

# 2.每个用户的平均客单价：每个用户总消费 ➗ 每个用户的总订单数
total_spent_count['avg_order_value'] = total_spent_count[
    'total_spent']/total_spent_count['order_count']
total_spent_count.head()

,customer_id,total_spent,order_count,avg_order_value
0,00012a2ce6f8dcda20d059ce98491703,114.74,1,114.74
1,000161a058600d5901f007fab4c27140,67.41,1,67.41
2,0001fd6190edaaf884bcaf3d49edf079,195.42,1,195.42
3,0002414f95344307404f0ace7a26f1d5,179.35,1,179.35
4,000379cdec625522490c315e70c7a9fb,107.01,1,107.01


#### 结论：用户平均客单价与用户订单总价大部分完全相同，因为绝大多数用户只下过一次单。

### 2.3.3每个用户的首次/最后一次购买时间

In [343]:
# 每个用户的首次/最后一次购买时间
buy_time = order_merge.groupby('customer_id')[
    'order_purchase_timestamp'].agg([('first_buy', 'min'),('last_buy', 'max')]).reset_index()
buy_time.head()

,customer_id,first_buy,last_buy
0,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,2017-11-14 16:08:26
1,000161a058600d5901f007fab4c27140,2017-07-16 09:40:32,2017-07-16 09:40:32
2,0001fd6190edaaf884bcaf3d49edf079,2017-02-28 11:06:43,2017-02-28 11:06:43
3,0002414f95344307404f0ace7a26f1d5,2017-08-16 13:09:20,2017-08-16 13:09:20
4,000379cdec625522490c315e70c7a9fb,2018-04-02 13:42:17,2018-04-02 13:42:17


#### 结论：每个用户的首次 / 最后一次购买时间完全相同,因为在这个数据集中：绝大多数用户只下过 1 次单

### 2.3.4合并且保存用户维度特征表
用于后续分析

In [344]:
user_features = cust_count.merge(total_spent_count[[
    'customer_id', 'total_spent', 'avg_order_value']], on='customer_id', how='left')

user_features = user_features.merge(buy_time, on='customer_id', how='left')

user_features.rename(columns={'order_id':'order_count'}, inplace=True)
user_features.head(3)

,customer_id,order_count,is_repeat,total_spent,avg_order_value,first_buy,last_buy
0,00012a2ce6f8dcda20d059ce98491703,1,False,114.74,114.74,2017-11-14 16:08:26,2017-11-14 16:08:26
1,000161a058600d5901f007fab4c27140,1,False,67.41,67.41,2017-07-16 09:40:32,2017-07-16 09:40:32
2,0001fd6190edaaf884bcaf3d49edf079,1,False,195.42,195.42,2017-02-28 11:06:43,2017-02-28 11:06:43


In [345]:
#保存用户维度特征表为csv
user_features.to_csv('brazil_ecommerce_project/features/user_features.csv', index=False)

### 结论：用户维度特征完整，逻辑严谨，结论可信。

#####

## 2.4构建订单维度特征

### 2.4.1每个订单的总金额

In [346]:
# 先分组求和，得到两列独立的求和结果
order_total_prices = order_merge.groupby('order_id')[['price' ,'freight_value']].sum().reset_index()

# 再计算每个订单的总金额（价格和 + 运费和）
order_total_prices['order_total_price'] = round((order_total_prices['price'] + order_total_prices[
    'freight_value']), 2)
order_total_prices.head()

,order_id,price,freight_value,order_total_price
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04


### 2.4.2每个订单的商品件数

In [347]:
order_total_counts = order_merge.groupby('order_id')['order_item_id'].count().reset_index()
# 或 order_merge.groupby('order_id').size()
order_total_counts.head()

,order_id,order_item_id
0,00010242fe8c5a6d1ba2dd792cb16214,1
1,00018f77f2f0320c557190d7a144bdd3,1
2,000229ec398224ef6ca0657da4fc703e,1
3,00024acbcdf0a6daa1e931b038114c75,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1


### 2.4.3每个订单的配送时长

In [348]:
# 实际送达时间 — 购买时间 = 配送时长
order_merge['delivery_days'] = (order_merge['order_delivered_customer_date'] - order_merge[
    'order_purchase_timestamp']).dt.days

# 聚合到订单粒度后输出
order_delivery = order_merge.groupby('order_id')['delivery_days'].first().reset_index()
order_delivery.head()

,order_id,delivery_days
0,00010242fe8c5a6d1ba2dd792cb16214,7.0
1,00018f77f2f0320c557190d7a144bdd3,16.0
2,000229ec398224ef6ca0657da4fc703e,7.0
3,00024acbcdf0a6daa1e931b038114c75,6.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,25.0


### 2.4.4每个订单的延迟情况

In [349]:
# 用布尔特征标记 订单的送达时间 > 预计送达时间 的情况
order_merge['is_late'] = order_merge['order_delivered_customer_date'] > order_merge[
    'order_estimated_delivery_date']

# 聚合到订单粒度后输出
late_time = order_merge.groupby('order_id')['is_late'].first().reset_index()
late_time.head()

,order_id,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,False
1,00018f77f2f0320c557190d7a144bdd3,False
2,000229ec398224ef6ca0657da4fc703e,False
3,00024acbcdf0a6daa1e931b038114c75,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,False


In [350]:
# 输出延迟率统计
late_rate = round(late_time['is_late'].mean() * 100, 2)
late_rate_str = f"{late_rate}%"
late_rate_str

'7.87%'

#### 结论：订单整体延迟率为7.87%

### 2.4.5合并且保存订单维度特征表
用于后续分析

In [351]:
order_features = order_total_prices[['order_id', 'order_total_price']].merge(
    order_total_counts, on='order_id', how='left')

order_features = order_features.merge(order_delivery, on='order_id', how='left')

order_features = order_features.merge(late_time, on='order_id', how='left')

order_features.rename(columns={'order_item_id':'order_count'}, inplace=True)

order_features.head()

,order_id,order_total_price,order_count,delivery_days,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,7.0,False
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,16.0,False
2,000229ec398224ef6ca0657da4fc703e,216.87,1,7.0,False
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,6.0,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,25.0,False


In [352]:
#保存订单维度特征表为csv
order_features.to_csv('brazil_ecommerce_project/features/order_features.csv', index=False)

### 结论：订单维度特征覆盖了金额、件数、时效、延迟，适合后续分析订单履约效率。

#####

## 2.5构建产品维度特征

### 2.5.1每个类别产品的销售额（不含运费）和销量
销售额（不含运费）反映产品本身的市场价值，用于判断哪些品类最赚钱、销售贡献最大

In [525]:
#以类别产品分组，计算price列的和，计算order_item_id列的数量，分别设置成更直观的字段名，然后充值索引。
category_parice_count = product_seller_merge.groupby(
    'product_category_name').agg(category_total_sales=('price', 'sum'),
                                 category_total_count=('order_item_id', 'count')).reset_index()
# 价格列保留2位小数
category_parice_count['category_total_sales'] = round(category_parice_count[
    'category_total_sales'],2)
category_parice_count.head()

,product_category_name,category_total_sales,category_total_count
0,agro_industria_e_comercio,72530.47,212
1,alimentos,29393.41,510
2,alimentos_bebidas,15179.48,278
3,artes,24202.64,209
4,artes_e_artesanato,1814.01,24


### 2.5.2每个类别产品的平均评分

In [526]:
# 先关联订单评分表
product_seller_reviews = product_seller_merge.merge(order_reviews, on='order_id', how='left')

# 让每个产品在每笔订单中只贡献一次评分，按订单-产品去重（每个订单每个产品只保留一行）
product_seller_reviews = product_seller_reviews.drop_duplicates(subset=['order_id', 'product_id'])

# 再按类别求平均评分
category_avg_score = round(product_seller_reviews.groupby('product_category_name')[
    'review_score'].mean(),1).reset_index()

# 修改平均评分字段名
category_avg_score.rename(columns={'review_score':'category_avg_score'}, inplace=True)
category_avg_score.head()

,product_category_name,category_avg_score
0,agro_industria_e_comercio,4.0
1,alimentos,4.3
2,alimentos_bebidas,4.4
3,artes,4.0
4,artes_e_artesanato,4.1


### 2.5.3每个产品类别中销售额前10的城市

In [527]:
# 连接订单表
category_city_merge = product_seller_merge.merge(orders[[
    'order_id', 'customer_id']], on='order_id', how='left')
# 连接用户表
category_city_merge = category_city_merge.merge(customers[[
    'customer_id', 'customer_city']], on='customer_id', how='left')

# 分组求和，得到每个类别在每个城市的总销售额
category_city_tops = category_city_merge.groupby([
    'product_category_name', 'customer_city']).agg(category_city_total_price=(
    'price', 'sum')).reset_index()

# 价格列保留2位小数
category_city_tops['category_city_total_price'] = round(category_city_tops[
    'category_city_total_price'],2)

# 用 idxmax() 找出销售额最高的城市行
idx_max = category_city_tops.groupby('product_category_name')[
    'category_city_total_price'].idxmax()
category_city_tops = category_city_tops.loc[idx_max]

# 对销售额降序排序，取前10
category_city_top = category_city_tops.sort_values(
    'category_city_total_price', ascending=False)[:10]
category_city_top

,product_category_name,customer_city,category_city_total_price
4130,beleza_saude,sao paulo,189361.87
6501,cama_mesa_banho,sao paulo,170705.41
22543,relogios_presentes,sao paulo,165897.46
15683,informatica_acessorios,sao paulo,144635.68
12268,esporte_lazer,sao paulo,144533.26
25582,utilidades_domesticas,sao paulo,104820.45
18336,moveis_decoracao,sao paulo,100033.05
5097,brinquedos,sao paulo,69635.89
9404,cool_stuff,sao paulo,67860.87
20810,perfumaria,sao paulo,62195.09


### 2.5.4合并且保存产品维度特征表
用于后续分析

In [528]:
category_merge = category_parice_count.merge(
    category_avg_score, on='product_category_name', how='left')

# 产品类别中销售额排名表，按照销售额降序取10上一步的表连接
category_features = category_merge.merge(category_city_tops, on='product_category_name', how='left')
category_features.head()

,product_category_name,category_total_sales,category_total_count,category_avg_score,customer_city,category_city_total_price
0,agro_industria_e_comercio,72530.47,212,4.0,sao paulo,10463.16
1,alimentos,29393.41,510,4.3,sao paulo,6142.41
2,alimentos_bebidas,15179.48,278,4.4,sao paulo,2957.15
3,artes,24202.64,209,4.0,marilia,6499.00
4,artes_e_artesanato,1814.01,24,4.1,brusque,289.49


In [529]:
# 保存产品维度特征表为csv
category_features.to_csv('brazil_ecommerce_project/features/category_features.csv', index=False)

### 结论：产品维度特征完整，去重逻辑严谨，适合品类竞争力分析。

#####

## 2.6特征工程总结
- 用户维度：统计了订单数、消费额、客单价及首次/最后一次购买时间。绝大多数用户仅下单一次，复购率极低。
- 订单维度：计算了订单总金额、商品件数、配送时长及是否延迟。整体订单延迟率为7.87%。
- 产品维度：分析了各品类销售额、销量、平均评分，并找出每个品类销售额最高的城市。圣保罗是绝大多数品类的消费中心。
- 所有特征表已保存，数据质量符合预期，可进入可视化分析阶段。